In [77]:
# Import Dependencies
import os
import certifi
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [78]:
from langchain.agents import create_react_agent,AgentExecutor

In [79]:
# Load Env Variable
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHER_STACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [80]:
# Search Tool
search_tool = TavilySearchResults(max_results=2)

In [81]:
result = search_tool.invoke("What is the capital of France")
result

[{'url': 'https://en.wikipedia.org/wiki/Paris',
  'content': 'As the capital of France, Paris is the seat of France\'s national government. For the executive, the two chief officers each have their own official residences, which also serve as their offices. The President of the French Republic resides at the Élysée Palace.( The Prime Minister\'s seat is at the Hôtel Matignon.( Government ministries are located in various parts of the city, many near the Hôtel Matignon.( [...] Paris( is the capital and largest city of France, with an estimated city population of 2.04 million in an area of 105.4 km 2 (40.7 sq mi), and a metropolitan population of 13.2 million as of January 2026( Located on the river Seine in the centre of the Île-de-France region, it is the largest metropolitan area and fourth-most populous city in the European Union (EU). Nicknamed the "City of Light", partly because of its role in the Age of Enlightenment, Paris has been one of the world\'s major centres of finance, di

In [83]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHER_STACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [84]:
llm = ChatGroq(
    model="qwen/qwen3-32b",
    api_key=GROQ_API_KEY,
    temperature=0)

In [85]:
response = llm.invoke("Can you give a joke about AI?")
response

AIMessage(content='<think>\nOkay, the user wants a joke about AI. Let me think about how to approach this. First, I need to make sure the joke is appropriate and not offensive. AI jokes can sometimes touch on sensitive topics, so I have to be careful.\n\nHmm, maybe play on common AI stereotypes or situations. Like, maybe something about machine learning or data training? Or perhaps a pun involving "algorithms" or "neural networks"? Let me brainstorm a few ideas.\n\nWait, there\'s a classic joke structure: "Why did the AI...? Because it needed to..." Maybe something about overfitting data? Or maybe a joke about AI misunderstanding human commands? Like, if you tell an AI to do something, it takes it literally?\n\nAnother angle: AI trying to sound human but failing. For example, "Why did the AI cross the road?" Then the punchline could be something about processing data or optimizing a path. Let me try to put that together.\n\nOriginal joke structure: "Why did the chicken cross the road? 

In [86]:
# PROMPT

prompt = hub.pull("hwchase17/react")

d:\AI ML Engineering\Langchain Agent\.venv\Lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [87]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [88]:
# TOOL
tools = [search_tool,get_weather_data]

In [89]:
# Create Agent
agent = create_react_agent(
    llm=llm,
    tools= tools,
    prompt= prompt
)

In [93]:
# EXECUTOR

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True
)

In [94]:
# RUN

response = agent_executor.invoke({
    "input": "Find The Capital Of Pakistan"
    "and finds its current weather"
})



> Entering new AgentExecutor chain...
<think>
Okay, let's tackle this question. The user wants to find the capital of Pakistan and then get its current weather. First, I need to confirm what the capital of Pakistan is. I remember that Islamabad is the capital, but maybe I should double-check to be sure. Sometimes people might confuse it with Karachi, which is the largest city. But I'm pretty sure Islamabad is correct.

So, first step: Use the tavily_search_results_json tool to search for "Capital of Pakistan" to confirm. That way, I can get an accurate answer. Once I have the capital, I can then use the get_weather_data tool to fetch the weather for that city. Let me start with the search.

After confirming the capital, I'll input that city into the weather function. Let me make sure the tools are used correctly. The search should return Islamabad, then I'll get the weather data for Islamabad. I need to present both the capital and the weather information as the final answer.
</think

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01kec8v6wheq58x1sn4p0zyy04` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 2814, Requested 3972. Please try again in 7.86s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [92]:
print(response["output"])

TypeError: 'AIMessage' object is not subscriptable